# 03 — Feature Importance (SHAP)

Uses `src.xai.shap_tree` to compute SHAP TreeExplainer attributions for the Random-Forest baseline trained in `runs/random_forest_cwru/` (regenerate via `make train-classical` if that directory is missing — it's gitignored except for `metrics.json`/`config.yaml`).


In [1]:
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.data.dataset import FAULT_CLASSES
from src.features.extract import CWRU_METADATA_COLUMNS
from src.models.classical import ClassicalFaultClassifier
from src.xai.shap_tree import explain_tree_model, top_features_for_class


Z:\turboguard\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
clf = ClassicalFaultClassifier.load('../runs/random_forest_cwru/model.joblib')
df = pd.read_parquet('../data/processed/cwru/features.parquet')
feature_cols = [c for c in df.columns if c not in CWRU_METADATA_COLUMNS]
X = df[feature_cols].to_numpy(dtype=np.float64)
shap_values = explain_tree_model(clf, X)
shap_values.shape


(20, 176, 5)

In [3]:
outer_race_idx = FAULT_CLASSES.index('outer_race')
top = top_features_for_class(shap_values, feature_cols, class_idx=outer_race_idx, top_n=10)

names, vals = zip(*top[::-1])
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(names, vals)
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('Top 10 features — outer-race fault class')
fig.tight_layout()
